# Model Evaluation Deep Dive — AI Lab Instructor Notebook

*Evaluation & Debugging · Medium*

Master evaluation choices under class imbalance and costs: ROC vs PR, thresholds, calibration, and confidence intervals.

**Outcome.** Students can:
- compute core metrics correctly,
- choose metrics aligned with product goals,
- tune decision thresholds using costs,
- diagnose calibration and improve it.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Model Evaluation Deep Dive — Student Lab

You will compute metrics, compare ROC vs PR, do cost-based thresholding, and evaluate calibration.

## Section 0 — Dataset

Default: synthetic imbalanced classification. Optional: Titanic for realism.

# TODO: set `use_titanic = False` or True

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

rng = np.random.default_rng(0)

## Task 2: Section 0 — Dataset
*Section: Dataset*

## Section 0 — Dataset

Default: synthetic imbalanced classification. Optional: Titanic for realism.

In [ ]:
use_titanic = False

def load_titanic_or_none():
    path = os.path.join(os.getcwd(), 'data', 'titanic', 'train.csv')
    if os.path.exists(path):
        return pd.read_csv(path)
    return None

if use_titanic:
    df = load_titanic_or_none()
    if df is None:
        use_titanic = False

if not use_titanic:
    X, y = make_classification(n_samples=5000, n_features=20, n_informative=6, n_redundant=4, weights=[0.9,0.1], flip_y=0.02, class_sep=1.0, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
else:
    df = df.dropna(subset=['Survived'])
    y = df['Survived'].astype(int).values
    X = pd.get_dummies(df[['Pclass','Sex','Age','Fare']].copy(), columns=['Sex'], drop_first=True)
    X['Age'] = X['Age'].fillna(X['Age'].median())
    X['Fare'] = X['Fare'].fillna(X['Fare'].median())
    Xtr, Xte, ytr, yte = train_test_split(X.values, y, test_size=0.3, random_state=0, stratify=y)

print('base_rate', float(np.mean(ytr)))

In [ ]:
# Checks
check('train_nonempty', len(ytr) > 0)

## Task 3: Implement confusion-matrix metrics
*Section: Metrics from scratch*

## Section 1 — Metrics from scratch

### Task 1.1: Implement confusion-matrix metrics

Given `y_true` and `y_pred` (0/1), implement:
- TP, FP, TN, FN
- precision, recall, f1
- TPR, FPR


**Checkpoint:** When is accuracy misleading?

In [ ]:
$10

## Task 4: Train LogisticRegression and compute ranking metrics
*Section: Train a model and compare ROC-AUC vs PR-AUC*

## Section 2 — Train a model and compare ROC-AUC vs PR-AUC

### Task 2.1: Train LogisticRegression and compute ranking metrics

- Fit logistic regression
- Compute ROC-AUC and PR-AUC on test
- Explain why PR-AUC is more informative under imbalance

In [ ]:
clf = LogisticRegression(max_iter=5000)
clf.fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]
print('roc_auc', roc_auc_score(yte, p))
print('pr_auc', average_precision_score(yte, p))

# Rationale: ROC can look good under severe imbalance; PR reflects precision at useful recall.

## Task 5: Choose threshold to minimize expected cost
*Section: Thresholding with costs*

## Section 3 — Thresholding with costs

### Task 3.1: Choose threshold to minimize expected cost

Let cost(FN)=10, cost(FP)=1.

- Sweep thresholds
- Compute expected cost
- Report best threshold

**Interview Angle:** how does the best threshold move when FN cost increases?

In [ ]:
cost_fp = 1.0
cost_fn = 10.0
thresholds = np.linspace(0.01, 0.99, 99)
best = None
for t in thresholds:
    yhat = (p >= t).astype(int)
    tp, fp, tn, fn = confusion_counts(yte, yhat)
    cost = cost_fp * fp + cost_fn * fn
    if best is None or cost < best[0]:
        best = (cost, float(t), int(fp), int(fn))
print('best (cost, thr, fp, fn):', best)

# Rationale: higher FN cost pushes threshold lower to catch more positives.

## Task 6: Implement reliability bins and ECE
*Section: Calibration: Reliability + ECE*

## Section 4 — Calibration: Reliability + ECE

### Task 4.1: Implement reliability bins and ECE

ECE (Expected Calibration Error) is a weighted avg of |avg_confidence - accuracy| per bin.

In [ ]:
$11

## Task 7: Temperature scaling (binary, post-hoc)
*Section: Calibration: Reliability + ECE*

### Task 4.2: Temperature scaling (binary, post-hoc)

We will learn a temperature `T>0` to calibrate logits: sigmoid(logit / T).

- Convert probabilities to logits
- Search over T grid to minimize NLL on a validation split
- Compare ECE before/after

In [ ]:
$12

## Task 8: Bootstrap a CI for PR-AUC
*Section: Bootstrap CI (uncertainty)*

## Section 5 — Bootstrap CI (uncertainty)

### Task 5.1: Bootstrap a CI for PR-AUC

- Sample indices with replacement
- Compute PR-AUC each bootstrap
- Report 2.5 and 97.5 percentiles

In [ ]:
# Bootstrap CI for PR-AUC
B = 500
vals = []
n = len(yte)
for _ in range(B):
    idx = rng.integers(0, n, size=n)
    vals.append(average_precision_score(yte[idx], p[idx]))
lo, hi = np.percentile(vals, [2.5, 97.5])
print('PR-AUC point:', average_precision_score(yte, p))
print('95% CI:', (float(lo), float(hi)))

# Rationale: point estimates hide variance; CI tells you if improvements are meaningful.